**Install Libraries**

In [1]:
!pip uninstall -y langchain langchain-core langchain-community pydantic transformers
!pip install langchain langchain-core langchain-community transformers faiss-cpu

Found existing installation: langchain 1.2.14
Uninstalling langchain-1.2.14:
  Successfully uninstalled langchain-1.2.14
Found existing installation: langchain-core 1.2.23
Uninstalling langchain-core-1.2.23:
  Successfully uninstalled langchain-core-1.2.23
Found existing installation: pydantic 2.12.3
Uninstalling pydantic-2.12.3:
  Successfully uninstalled pydantic-2.12.3
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.7/508.7 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 93.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

**BASIC LLM**

In [1]:
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

pipe = pipeline("text-generation", model="gpt2", max_length=100)
llm = HuggingFacePipeline(pipeline=pipe)

print(llm.invoke("What is Artificial Intelligence?"))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
/tmp/ipykernel_14051/1986659797.py:5: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


What is Artificial Intelligence?

In this post I will try to introduce some of the most interesting aspects of Artificial Intelligence (AI) and how it is developed.

We will be looking at how AI is built. If you want to learn


**PROMPT TEMPLATE**

In [2]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms"
)

formatted = prompt.format(topic="Machine Learning")
print(formatted)

Explain Machine Learning in simple terms


CHAIN

In [3]:
chain = prompt | llm

output = chain.invoke({"topic": "LangChain"})
print(output)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Explain LangChain in simple terms

3.2.2.3.3.3.3.3.3.3.3.3.3.3.3.3.3.3.3.3.3


MEMORY

In [6]:

class SimpleMemory:
    def __init__(self):
        self.history = []

    def save_context(self, user_input, ai_output):
        self.history.append({
            "input": user_input,
            "output": ai_output
        })

    def load_memory(self):
        return self.history


# Using memory
memory = SimpleMemory()

memory.save_context("Hi", "Hello")
memory.save_context("What is AI?", "AI is Artificial Intelligence")

print(memory.load_memory())

[{'input': 'Hi', 'output': 'Hello'}, {'input': 'What is AI?', 'output': 'AI is Artificial Intelligence'}]


**AGENT + TOOL**

In [7]:


from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

# Load model
pipe = pipeline("text-generation", model="gpt2", max_length=100)
llm = HuggingFacePipeline(pipeline=pipe)

# Define a simple tool (manual function)
def multiply_numbers(text):
    nums = [int(s) for s in text.split() if s.isdigit()]
    if len(nums) >= 2:
        return str(nums[0] * nums[1])
    return "Please provide two numbers"

# Simulated agent logic (safe alternative)
def simple_agent(query):
    if "multiply" in query.lower():
        return multiply_numbers(query)
    else:
        return llm.invoke(query)

# Test
print(simple_agent("multiply 4 and 5"))
print(simple_agent("What is AI?"))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


20
What is AI?

AI is the ability to create worlds to interact with each other in a virtual world, and it can be used to create games or interact with other NPCs in a virtual world. It's also the ability to create interactive worlds


**DOCUMENT LOADER**

In [8]:
# Simple Document Loader (no dependency issues)

# Create sample file
with open("sample.txt", "w") as f:
    f.write("LangChain is a framework for building LLM applications.")

def load_document(file_path):
    with open(file_path, "r") as f:
        return f.read()

doc = load_document("sample.txt")

print(doc)

LangChain is a framework for building LLM applications.


**VECTOR STORE**

In [9]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Data
texts = [
    "LangChain is powerful",
    "AI is the future",
    "Machine learning is part of AI"
]

# Convert to vectors
vectorizer = TfidfVectorizer()
vectors = vectorizer.fit_transform(texts)

# Search function
def search(query):
    query_vec = vectorizer.transform([query])
    similarity = cosine_similarity(query_vec, vectors)
    index = similarity.argmax()
    return texts[index]

# Test
print(search("What is LangChain?"))

LangChain is powerful
